# 🧹 EduNilai — Data Integrity Check & Cleaning

**Stage:** 01b — Data Cleaning (runs before EDA)
**Output:** Cleaned CSVs saved to `../data/cleaned/`

## Issues Found (Pre-Inspection Summary)

| Dataset | Issue | Action |
|---|---|---|
| `fees_raw` | 3 duplicate rows (exact programme + fee match) | Drop duplicates |
| `fees_raw` | Column name `fee` — rename for clarity | Rename to `total_fee` |
| `fees_mean` | NaN in UTM (3 fields) and APU (4 fields) — universities don't offer those fields | Keep as NaN — structurally missing, not errors |
| `hh_income_percentiles` | 6 NaN rows — percentile 1 minimum & percentile 100 maximum across all years | Drop — boundary values undefined by DOSM |
| `asean` | `human_capital_index` — 107/140 nulls (most countries/years missing) | Drop column — too sparse to use |
| `asean` | `unemployment_advanced_edu_pct` — 50 nulls (mainly small countries) | Keep — Malaysia, Thailand, Indonesia complete |
| All date cols | Loaded as string — need `parse_dates` | Convert to datetime |
| `lfs` | `sex` uses `both` instead of `overall` | Standardise to `overall` |


## ⚙️ Setup

In [2]:
import pandas as pd
import numpy as np
import os

os.makedirs('../data/cleaned', exist_ok=True)
print("Output folder ready: ../data/cleaned/")


Output folder ready: ../data/cleaned/


---
## 1. `fees_raw` — Programme-Level Tuition Fees

**Issues:**
- 3 exact duplicate rows (same university + programme + fee)
- Column `fee` → rename to `total_fee` for clarity


In [5]:
fees_raw = pd.read_csv('../data/cost/edunilai_raw_fees.csv')

print(f"Shape before: {fees_raw.shape}")
print(f"Duplicates: {fees_raw.duplicated().sum()}")
print("\nDuplicate rows:")
fees_raw[fees_raw.duplicated(keep=False)][['university','field_category','programme','fee']]


Shape before: (189, 5)
Duplicates: 3

Duplicate rows:


,university,field_category,programme,fee
115,APU,Computer Science & IT,BSc (Hons) in Information Technology with a sp...,103000.0
119,APU,Computer Science & IT,BSc (Hons) in Information Technology with a sp...,103000.0
144,Taylor's,Computer Science & IT,Bachelor Of Information Technology (Hons) (3+0...,121730.0
145,Taylor's,Arts & Humanities,Bachelor Of Mass Communication (Hons),120650.0
158,Taylor's,Arts & Humanities,Bachelor Of Mass Communication (Hons),120650.0
164,Taylor's,Computer Science & IT,Bachelor Of Information Technology (Hons) (3+0...,121730.0


In [6]:
# Fix 1: drop duplicates (keep first occurrence)
fees_raw = fees_raw.drop_duplicates().reset_index(drop=True)

# Fix 2: rename fee column
fees_raw = fees_raw.rename(columns={'fee': 'total_fee'})

print(f"Shape after:  {fees_raw.shape}")
print(f"Duplicates remaining: {fees_raw.duplicated().sum()}")
print(f"Columns: {list(fees_raw.columns)}")
print(f"\nFee stats:")
print(fees_raw['total_fee'].describe().round(0))


Shape after:  (186, 5)
Duplicates remaining: 0
Columns: ['university', 'type', 'field_category', 'programme', 'total_fee']

Fee stats:
count       186.0
mean      70113.0
std       61906.0
min        7710.0
25%        9700.0
50%      100600.0
75%      115099.0
max      446690.0
Name: total_fee, dtype: float64


In [7]:
fees_raw.to_csv('../data/cleaned/fees_raw.csv', index=False, encoding='utf-8')
print("Saved -> ../data/cleaned/fees_raw.csv")
print(f"Rows: {len(fees_raw)}")


Saved -> ../data/cleaned/fees_raw.csv
Rows: 186


---
## 2. `fees_mean` — Mean Fees by Field × University

**Issues:**
- NaN in UTM for: Arts & Humanities, Law, Medicine & Health (UTM does not offer these)
- NaN in APU for: Education, Law, Medicine & Health, Science (APU does not offer these)
- These are **structurally missing** — not data errors. We keep them as NaN and document clearly.


In [8]:
fees_mean = pd.read_csv('../data/cost/edunilai_mean_fees.csv')

print("NaN pattern — which uni does NOT offer which field:")
null_mask = fees_mean[['field_category','UTM','UM','APU',"Taylor's"]].isnull()
for _, row in fees_mean.iterrows():
    missing = [u for u in ['UTM','UM','APU',"Taylor's"] if pd.isna(row[u])]
    if missing:
        print(f"  {row['field_category']:35s} → NOT offered by: {missing}")


NaN pattern — which uni does NOT offer which field:
  Arts & Humanities                   → NOT offered by: ['UTM']
  Education                           → NOT offered by: ['APU']
  Law                                 → NOT offered by: ['UTM', 'APU']
  Medicine & Health                   → NOT offered by: ['UTM', 'APU']
  Science                             → NOT offered by: ['APU']


In [9]:
# No rows to drop — NaNs are structural (university does not offer the field)
# Document the Uni_Count column which already tracks this correctly
print("Uni_Count verification:")
print(fees_mean[['field_category','Uni_Count','Overall_Mean_RM']].to_string(index=False))
print("\nNote: Overall_Mean_RM is computed only from universities that offer the field.")
print("      This is correct — do NOT impute or fill these NaNs.")

fees_mean.to_csv('../data/cleaned/fees_mean.csv', index=False, encoding='utf-8')
print("\nSaved -> ../data/cleaned/fees_mean.csv  (unchanged — NaNs are structural)")


Uni_Count verification:
                  field_category  Uni_Count  Overall_Mean_RM
            Accounting & Finance          4         66233.54
Architecture & Built Environment          4         61747.50
               Arts & Humanities          3         76426.68
                        Business          4         59020.72
           Computer Science & IT          4         62805.00
                       Education          3         41559.44
                     Engineering          4         82716.25
                             Law          2         67617.50
               Medicine & Health          2        109638.50
                         Science          3         47905.56
                 Social Sciences          4         56946.67

Note: Overall_Mean_RM is computed only from universities that offer the field.
      This is correct — do NOT impute or fill these NaNs.

Saved -> ../data/cleaned/fees_mean.csv  (unchanged — NaNs are structural)


---
## 3. `cpi` — Consumer Price Index

**Issues:** None. Parse date column to datetime.


In [10]:
cpi = pd.read_csv('../data/inflation/cpi_2010_onwards.csv', parse_dates=['date'])

print(f"Shape: {cpi.shape}")
print(f"Date range: {cpi['date'].min().date()} to {cpi['date'].max().date()}")
print(f"Divisions: {sorted(cpi['division'].unique())}")
print(f"Nulls: {cpi.isnull().sum().sum()}")
print(f"Duplicates: {cpi.duplicated().sum()}")


Shape: (224, 3)
Date range: 2010-01-01 to 2025-01-01
Divisions: ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', 'overall']
Nulls: 0
Duplicates: 0


In [11]:
cpi.to_csv('../data/cleaned/cpi.csv', index=False, encoding='utf-8')
print("Saved -> ../data/cleaned/cpi.csv")


Saved -> ../data/cleaned/cpi.csv


---
## 4. `lfs_annual_sex` — Annual Labour Force Statistics by Sex

**Issues:**
- `sex` column uses `both` — standardise to `overall` to match other datasets (sru_sex uses `overall`)
- Parse date to datetime


In [12]:
lfs = pd.read_csv('../data/salary/lfs_annual_sex.csv', parse_dates=['date'])

print("Before — sex categories:", lfs['sex'].unique())
print(f"Shape: {lfs.shape}")
print(f"Date range: {lfs['date'].min().date()} to {lfs['date'].max().date()}")


Before — sex categories: ['both' 'female' 'male']
Shape: (42, 9)
Date range: 2010-01-01 to 2023-01-01


In [13]:
# Fix: standardise 'both' -> 'overall'
lfs['sex'] = lfs['sex'].replace({'both': 'overall'})
print("After — sex categories:", lfs['sex'].unique())

# Verify no nulls
print(f"Nulls: {lfs.isnull().sum().sum()}")

lfs.to_csv('../data/cleaned/lfs_annual_sex.csv', index=False, encoding='utf-8')
print("Saved -> ../data/cleaned/lfs_annual_sex.csv")


After — sex categories: ['overall' 'female' 'male']
Nulls: 0
Saved -> ../data/cleaned/lfs_annual_sex.csv


---
## 5. `graduate_underemployment_age` — SRU by Age Group

**Issues:** None. Parse date, verify coverage.


In [14]:
sru_age = pd.read_csv('../data/salary/graduate_underemployment_age.csv', parse_dates=['date'])

print(f"Shape: {sru_age.shape}")
print(f"Date range: {sru_age['date'].min().date()} to {sru_age['date'].max().date()}")
print(f"Age groups: {sorted(sru_age['age'].unique())}")
print(f"Rows per age group: {sru_age['age'].value_counts().to_dict()}")
print(f"Nulls: {sru_age.isnull().sum().sum()}")

# Verify quarterly frequency
quarters_per_age = sru_age.groupby('age')['date'].nunique()
print(f"\nQuarters per age group:")
print(quarters_per_age.to_string())


Shape: (175, 4)
Date range: 2017-01-01 to 2025-07-01
Age groups: ['15-24', '25-34', '35-44', '45+', 'overall']
Rows per age group: {'15-24': 35, '25-34': 35, '35-44': 35, '45+': 35, 'overall': 35}
Nulls: 0

Quarters per age group:
age
15-24      35
25-34      35
35-44      35
45+        35
overall    35


In [15]:
sru_age.to_csv('../data/cleaned/graduate_underemployment_age.csv', index=False, encoding='utf-8')
print("Saved -> ../data/cleaned/graduate_underemployment_age.csv")


Saved -> ../data/cleaned/graduate_underemployment_age.csv


---
## 6. `graduate_underemployment_sex` — SRU by Sex

**Issues:** Column named `sru_person` (missing 's') — rename to `sru_persons` to match `sru_age`.


In [16]:
sru_sex = pd.read_csv('../data/salary/graduate_underemployment_sex.csv', parse_dates=['date'])

print(f"Columns: {list(sru_sex.columns)}")
print(f"Shape: {sru_sex.shape}")
print(f"Sex categories: {sru_sex['sex'].unique()}")
print(f"Date range: {sru_sex['date'].min().date()} to {sru_sex['date'].max().date()}")


Columns: ['date', 'sex', 'sru_person', 'sru_rate']
Shape: (105, 4)
Sex categories: ['female' 'male' 'overall']
Date range: 2017-01-01 to 2025-07-01


In [17]:
# Fix: rename sru_person -> sru_persons for consistency with sru_age
sru_sex = sru_sex.rename(columns={'sru_person': 'sru_persons'})
print(f"Columns after rename: {list(sru_sex.columns)}")

sru_sex.to_csv('../data/cleaned/graduate_underemployment_sex.csv', index=False, encoding='utf-8')
print("Saved -> ../data/cleaned/graduate_underemployment_sex.csv")


Columns after rename: ['date', 'sex', 'sru_persons', 'sru_rate']
Saved -> ../data/cleaned/graduate_underemployment_sex.csv


---
## 7. `youth_unemployment_monthly` — Monthly Youth Unemployment

**Issues:** None. Parse date, verify no gaps.


In [18]:
youth = pd.read_csv('../data/salary/youth_unemployment_monthly.csv', parse_dates=['date'])

print(f"Shape: {youth.shape}")
print(f"Date range: {youth['date'].min().date()} to {youth['date'].max().date()}")

# Check for missing months
full_range = pd.date_range(youth['date'].min(), youth['date'].max(), freq='MS')
missing = full_range.difference(youth['date'])
print(f"Missing months: {len(missing)} {'(none)' if len(missing)==0 else missing.tolist()}")
print(f"Nulls: {youth.isnull().sum().to_dict()}")


Shape: (120, 5)
Date range: 2016-01-01 to 2025-12-01
Missing months: 0 (none)
Nulls: {'date': 0, 'unemployed_15_24': 0, 'u_rate_15_24': 0, 'unemployed_15_30': 0, 'u_rate_15_30': 0}


In [19]:
youth.to_csv('../data/cleaned/youth_unemployment_monthly.csv', index=False, encoding='utf-8')
print("Saved -> ../data/cleaned/youth_unemployment_monthly.csv")


Saved -> ../data/cleaned/youth_unemployment_monthly.csv


---
## 8. `hh_income_national` — Household Income National

**Issues:** None. HIES survey years only (not annual) — this is expected.


In [20]:
hh_nat = pd.read_csv('../data/demographic/hh_income_national.csv', parse_dates=['date'])

print(f"Shape: {hh_nat.shape}")
print(f"Survey years: {hh_nat['date'].dt.year.tolist()}")
print("Note: HIES is conducted every 2-3 years — gaps are expected, not errors.")
print(hh_nat.to_string(index=False))


Shape: (6, 3)
Survey years: [2012, 2014, 2016, 2019, 2020, 2022]
Note: HIES is conducted every 2-3 years — gaps are expected, not errors.
      date  income_mean  income_median
2012-01-01         5000           3626
2014-01-01         6141           4585
2016-01-01         6958           5228
2019-01-01         7901           5873
2020-01-01         7089           5209
2022-01-01         8479           6338


In [21]:
hh_nat.to_csv('../data/cleaned/hh_income_national.csv', index=False, encoding='utf-8')
print("Saved -> ../data/cleaned/hh_income_national.csv")


Saved -> ../data/cleaned/hh_income_national.csv


---
## 9. `hh_income_state` — Household Income by State

**Issues:** None. Parse date, verify state coverage.


In [22]:
hh_state = pd.read_csv('../data/demographic/hh_income_state.csv', parse_dates=['date'])

print(f"Shape: {hh_state.shape}")
print(f"States ({hh_state['state'].nunique()}): {sorted(hh_state['state'].unique())}")
print(f"Survey years: {sorted(hh_state['date'].dt.year.unique())}")
print(f"Nulls: {hh_state.isnull().sum().sum()}")

# Check all states present in all years
pivot_check = hh_state.pivot_table(index='state', columns=hh_state['date'].dt.year,
                                    values='income_median', aggfunc='count')
missing_combos = pivot_check.isnull().sum().sum()
print(f"\nMissing state-year combinations: {missing_combos}")


Shape: (96, 4)
States (16): ['Johor', 'Kedah', 'Kelantan', 'Melaka', 'Negeri Sembilan', 'Pahang', 'Perak', 'Perlis', 'Pulau Pinang', 'Sabah', 'Sarawak', 'Selangor', 'Terengganu', 'W.P. Kuala Lumpur', 'W.P. Labuan', 'W.P. Putrajaya']
Survey years: [2012, 2014, 2016, 2019, 2020, 2022]
Nulls: 0

Missing state-year combinations: 0


In [23]:
hh_state.to_csv('../data/cleaned/hh_income_state.csv', index=False, encoding='utf-8')
print("Saved -> ../data/cleaned/hh_income_state.csv")


Saved -> ../data/cleaned/hh_income_state.csv


---
## 10. `hh_income_percentiles` — Household Income by Percentile

**Issues:**
- 6 NaN rows: `variable == 'minimum'` at percentile 1, and `variable == 'maximum'` at percentile 100 across all 3 years
- These are undefined boundary values (the minimum income of the bottom 1% and maximum of the top 1% are not published by DOSM)
- **Action:** Drop these 6 rows


In [24]:
hh_pct = pd.read_csv('../data/demographic/hh_income_percentiles.csv', parse_dates=['date'])

print(f"Shape before: {hh_pct.shape}")
print(f"Null rows:")
print(hh_pct[hh_pct['income'].isnull()].to_string(index=False))


Shape before: (1200, 4)
Null rows:
      date  percentile variable  income
2024-01-01           1  minimum     NaN
2024-01-01         100  maximum     NaN
2022-01-01           1  minimum     NaN
2022-01-01         100  maximum     NaN
2019-01-01           1  minimum     NaN
2019-01-01         100  maximum     NaN


In [25]:
# Fix: drop undefined boundary NaN rows
hh_pct = hh_pct.dropna(subset=['income']).reset_index(drop=True)

print(f"Shape after: {hh_pct.shape}")
print(f"Nulls remaining: {hh_pct.isnull().sum().sum()}")
print(f"Variables: {hh_pct['variable'].unique()}")
print(f"Survey years: {sorted(hh_pct['date'].dt.year.unique())}")

hh_pct.to_csv('../data/cleaned/hh_income_percentiles.csv', index=False, encoding='utf-8')
print("Saved -> ../data/cleaned/hh_income_percentiles.csv")


Shape after: (1194, 4)
Nulls remaining: 0
Variables: ['mean' 'median' 'minimum' 'maximum']
Survey years: [2019, 2022, 2024]
Saved -> ../data/cleaned/hh_income_percentiles.csv


---
## 11. `income_inequality_gini` — Gini Coefficient

**Issues:** None. HIES years only — expected.


In [26]:
gini = pd.read_csv('../data/demographic/income_inequality_gini.csv', parse_dates=['date'])

print(f"Shape: {gini.shape}")
print(f"Survey years: {gini['date'].dt.year.tolist()}")
print(gini.to_string(index=False))

gini.to_csv('../data/cleaned/income_inequality_gini.csv', index=False, encoding='utf-8')
print("Saved -> ../data/cleaned/income_inequality_gini.csv")


Shape: (5, 2)
Survey years: [2012, 2014, 2016, 2019, 2022]
      date  gini
2012-01-01 0.431
2014-01-01 0.401
2016-01-01 0.399
2019-01-01 0.407
2022-01-01 0.404
Saved -> ../data/cleaned/income_inequality_gini.csv


---
## 12. `asean_comparison` — ASEAN World Bank Indicators

**Issues:**
- `human_capital_index`: 107 out of 140 rows are null (missing for most countries/years) — **drop column**, too sparse
- `unemployment_advanced_edu_pct`: 50 nulls — keep column, nulls concentrated in smaller countries (Cambodia, Laos, Myanmar); Malaysia, Thailand, Indonesia are complete
- Other nulls are structurally missing (small countries with limited World Bank data coverage) — **keep as NaN**, do not impute


In [27]:
asean = pd.read_csv('../data/asean/asean_comparison.csv')

print(f"Shape before: {asean.shape}")
print(f"\nNull counts per column:")
print(asean.isnull().sum())
print(f"\nhuman_capital_index coverage: {asean['human_capital_index'].notna().sum()}/140 rows")


Shape before: (140, 9)

Null counts per column:
country_code                         0
country                              0
year                                 0
unemployment_advanced_edu_pct       50
gdp_per_capita_usd                   0
education_expenditure_pct_gdp       27
labour_force_participation_rate      0
tertiary_gross_enrolment_ratio      20
human_capital_index                107
dtype: int64

human_capital_index coverage: 33/140 rows


In [28]:
# Fix: drop human_capital_index — too sparse (107/140 missing)
asean = asean.drop(columns=['human_capital_index'])

print(f"Shape after: {asean.shape}")
print(f"Columns: {list(asean.columns)}")
print(f"\nRemaining nulls:")
print(asean.isnull().sum())
print(f"\nMalaysia completeness:")
mys = asean[asean['country_code']=='MYS']
print(mys.isnull().sum())


Shape after: (140, 8)
Columns: ['country_code', 'country', 'year', 'unemployment_advanced_edu_pct', 'gdp_per_capita_usd', 'education_expenditure_pct_gdp', 'labour_force_participation_rate', 'tertiary_gross_enrolment_ratio']

Remaining nulls:
country_code                        0
country                             0
year                                0
unemployment_advanced_edu_pct      50
gdp_per_capita_usd                  0
education_expenditure_pct_gdp      27
labour_force_participation_rate     0
tertiary_gross_enrolment_ratio     20
dtype: int64

Malaysia completeness:
country_code                       0
country                            0
year                               0
unemployment_advanced_edu_pct      7
gdp_per_capita_usd                 0
education_expenditure_pct_gdp      0
labour_force_participation_rate    0
tertiary_gross_enrolment_ratio     0
dtype: int64


In [29]:
asean.to_csv('../data/cleaned/asean_comparison.csv', index=False, encoding='utf-8')
print("Saved -> ../data/cleaned/asean_comparison.csv")
print(f"Rows: {len(asean)}")


Saved -> ../data/cleaned/asean_comparison.csv
Rows: 140


---
## ✅ Cleaning Summary

All cleaned files are saved to `../data/cleaned/`. The EDA notebook (`02_eda.ipynb`) reads from this folder.


In [30]:
import os

cleaned_dir = '../data/cleaned'
files = os.listdir(cleaned_dir)
print(f"Files in ../data/cleaned/ ({len(files)} total):")
for f in sorted(files):
    path = os.path.join(cleaned_dir, f)
    df = pd.read_csv(path)
    print(f"  {f:45s} {df.shape}")

print("\nAll issues resolved:")
print("  fees_raw        — 3 duplicates dropped, 'fee' renamed to 'total_fee'")
print("  fees_mean       — NaNs kept (structural: university does not offer field)")
print("  lfs_annual_sex  — 'both' standardised to 'overall'")
print("  sru_sex         — 'sru_person' renamed to 'sru_persons'")
print("  hh_percentiles  — 6 undefined boundary NaN rows dropped")
print("  asean           — 'human_capital_index' column dropped (107/140 missing)")
print("  All others      — date parsed, no issues found")


Files in ../data/cleaned/ (12 total):
  asean_comparison.csv                          (140, 8)
  cpi.csv                                       (224, 3)
  fees_mean.csv                                 (11, 7)
  fees_raw.csv                                  (186, 5)
  graduate_underemployment_age.csv              (175, 4)
  graduate_underemployment_sex.csv              (105, 4)
  hh_income_national.csv                        (6, 3)
  hh_income_percentiles.csv                     (1194, 4)
  hh_income_state.csv                           (96, 4)
  income_inequality_gini.csv                    (5, 2)
  lfs_annual_sex.csv                            (42, 9)
  youth_unemployment_monthly.csv                (120, 5)

All issues resolved:
  fees_raw        — 3 duplicates dropped, 'fee' renamed to 'total_fee'
  fees_mean       — NaNs kept (structural: university does not offer field)
  lfs_annual_sex  — 'both' standardised to 'overall'
  sru_sex         — 'sru_person' renamed to 'sru_persons'
  hh